# **3 ВАРИАНТ** Танюшкин А.Л.

# Лабораторная работа № 2. Работа с mlflow
    Цель лабораторной работы – получение навыков логирования своей работы в mlflow.

1.	Использовать задание из вашего варианта первой лабораторной работы

✓

2.	Настроить окружение проекта локально на комьютере:

a.	Создать отдельную директорию под эту лабораторку

b.	Установить зависимости в отдельное виртуальное окружение для этой лабы

c.	Обязательно установить mlflow как зависимость

d.	Прописать зависимости лабораторной в файле requirements.txt (или использовать poetry)

e.	Скопировать ваш ноутбук из первой лабы в папку с лабораторной работой


a.	Создать отдельную директорию под эту лабораторку

✓

b.	Установить зависимости в отдельное виртуальное окружение для этой лабы

✓

c.	Обязательно установить mlflow как зависимость

✓

d.	Прописать зависимости лабораторной в файле requirements.txt (или использовать poetry)

✓

e.	Скопировать ваш ноутбук из первой лабы в папку с лабораторной работой

✓

3.	Далее необходимо запустить Mlflow Server

✓

4.	Ноутбук из своей первой лабораторной модифицируете по следующим правилам:

a.	Добавляете подключение к mlflow серверу

b.	Включаете автоматическое логирование экспериментов keras

i.	Логировать сами модели надо

ii.	Логировать датасеты не надо

c.	Делаете один прогон обучения с набором параметров – у вас появится новый ран в mlflow

d.	Копируете ID рана, который вы получили от mlflow. Вновь его запускаете и добавляете метрики качества, которые вы оцениваете на тестовом датасете. (пример см. здесь)

e.	Далее меняете набор параметров или архитектуру модели

f.	Повторяйте шаги C-E минимум 3 раза

g.	В результате вы получите минимум три рана в вашем сервере mlflow


In [56]:
import os
import mlflow
#import mlflow.tensorflow
#from mlflow.tensorflow import MlflowCallback
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout # type: ignore
import keras_tuner as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow

In [57]:
df = pd.read_csv('data.csv')

In [58]:
df.head(1)

,Unnamed: 0,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,ecology_excellent,ecology_good,ecology_no data,ecology_poor,ecology_satisfactory,price_doc
0,0,-0.294873,-0.2052,-0.690611,0.128662,-0.212332,-0.004459,-0.064771,-0.083784,0.393775,...,0.915176,-0.420245,-0.017208,-0.406425,-0.385252,1.80206,-0.579283,-0.59758,-0.370907,-0.266324


In [59]:
X = df.drop('price_doc', axis=1)
y = df['price_doc']

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=5
)

In [61]:
os.environ['USER'] = 'Aleggg'

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment('experiment_with_mlflow_lab2')

2025/11/09 16:50:56 INFO mlflow.tracking.fluent: Experiment with name 'experiment_with_mlflow_lab2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1762696256849, experiment_id='1', last_update_time=1762696256849, lifecycle_stage='active', name='experiment_with_mlflow_lab2', tags={}>

In [62]:
mlflow.keras.autolog(log_models=True,log_datasets=False)

In [63]:
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    return {
        "test_mse": mse,
        "test_mae": mae, 
        "test_r2": r2,
        "test_rmse": rmse
    }

In [64]:
def create_model(hidden_layers, units, dropout_rate, learning_rate):
    model = keras.Sequential()
    
    model.add(Dense(units, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(dropout_rate))
    
    for i in range(hidden_layers - 1):
        model.add(Dense(units // (2 ** (i + 1)), activation='relu'))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'])
    
    return model

In [65]:
def first_run(run_name='first_run'):
    with mlflow.start_run(run_name=run_name):
        params_1 = {
            'hidden_layers': 3,
            'units': 128,
            'dropout_rate': 0.3,
            'learning_rate': 0.001,
            'epochs': 50,
            'batch_size': 32
            }
        
        mlflow.log_params(params_1)

        model_1 = create_model(
            hidden_layers=params_1['hidden_layers'],
            units=params_1['units'],
            dropout_rate=params_1['dropout_rate'],
            learning_rate=params_1['learning_rate']
        )

        model_1.fit(X_train, y_train, epochs=params_1['epochs'],
                    batch_size=params_1['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_1.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [66]:
def second_run(run_name='second_run'):
    with mlflow.start_run(run_name=run_name):
        params_2 = {
            'hidden_layers': 4,
            'units': 256,
            'dropout_rate': 0.2,
            'learning_rate': 0.0005,
            'epochs': 30,
            'batch_size': 64
            }
        
        mlflow.log_params(params_2)

        model_2 = create_model(
            hidden_layers=params_2['hidden_layers'],
            units=params_2['units'],
            dropout_rate=params_2['dropout_rate'],
            learning_rate=params_2['learning_rate']
        )

        model_2.fit(X_train, y_train, epochs=params_2['epochs'],
                    batch_size=params_2['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_2.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [67]:
def third_run(run_name='third_run'):
    with mlflow.start_run(run_name=run_name):
        params_3 = {
            'hidden_layers': 4,
            'units': 512,
            'dropout_rate': 0.6,
            'learning_rate': 0.001,
            'epochs': 100,
            'batch_size': 64
            }
        
        mlflow.log_params(params_3)

        model_3 = create_model(
            hidden_layers=params_3['hidden_layers'],
            units=params_3['units'],
            dropout_rate=params_3['dropout_rate'],
            learning_rate=params_3['learning_rate']
        )

        model_3.fit(X_train, y_train, epochs=params_3['epochs'],
                    batch_size=params_3['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_3.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [68]:
first_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 53481.7344 - mae: 81.0780 - val_loss: 1.2320 - val_mae: 0.8374
Epoch 2/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 556.0565 - mae: 5.9000 - val_loss: 1.0057 - val_mae: 0.6174
Epoch 3/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 253.2513 - mae: 2.9593 - val_loss: 1.0070 - val_mae: 0.6156
Epoch 4/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 83.4815 - mae: 1.6566 - val_loss: 1.0072 - val_mae: 0.6148
Epoch 5/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 35.6656 - mae: 1.2096 - val_loss: 1.0072 - val_mae: 0.6150
Epoch 6/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 22.6687 - mae: 0.9821 - val_loss: 1.0072 - val_mae: 0.6145
Epoch 7/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 24.2231 - mae: 0.8980 - val_loss: 1.0072 - val_mae: 0.6147
Epoch 8/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.9734 - mae: 0.8336 - val_loss: 1.0072 - val_mae: 0.6146
Epoch 9/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 2s

2025/11/09 16:52:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:52:24 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during keras autologging: BAD_REQUEST: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: metrics.key, metrics.timestamp, metrics.step, metrics.run_uuid, metrics.value, metrics.is_nan
[SQL: INSERT INTO metrics ("key", value, timestamp, step, is_nan, run_uuid) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('loss', 53481.734375, 1762696259912, 0, 0, 'e506a205cd9142869de53c606ef26f77'), ('mae', 81.07804870605469, 1762696259912, 0, 0, 'e506a205cd9142869de53c606ef26f77'), ('val_loss', 1.2319830656051636, 1762696259912, 0, 0, 'e506a205cd9142869de53c606ef26f77'), ('val_mae', 0.8374060988426208, 1762696259912, 0, 0, 'e506a205cd9142869de53c606ef26f77')]]
(Background on this err

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step
🏃 View run first_run at: http://localhost:5000/#/experiments/1/runs/e506a205cd9142869de53c606ef26f77
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [69]:
second_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 36s 8ms/step - loss: 37182.0391 - mae: 89.5204 - val_loss: 11.8432 - val_mae: 2.9901
Epoch 2/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1326.5558 - mae: 21.7187 - val_loss: 12.4718 - val_mae: 3.0679
Epoch 3/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 368.5845 - mae: 9.6385 - val_loss: 1.0565 - val_mae: 0.7087
Epoch 4/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 174.1608 - mae: 5.0153 - val_loss: 1.0087 - val_mae: 0.6487
Epoch 5/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 94.5789 - mae: 3.1470 - val_loss: 1.0097 - val_mae: 0.6512
Epoch 6/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 60.1552 - mae: 2.1452 - val_loss: 1.0010 - val_mae: 0.6269
Epoch 7/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 46.5729 - mae: 1.7407 - val_loss: 1.0162 - val_mae: 0.6020
Epoch 8/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 28.2966 - mae: 1.3487 - val_loss: 1.0007 - val_mae: 0.6174
Epoch 9/30
381/381 ━━━━━━━━━━━━━━━━━

2025/11/09 16:53:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
🏃 View run second_run at: http://localhost:5000/#/experiments/1/runs/2713fdbcc7e349a2a9a505f7a841872e
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [70]:
third_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - loss: 356306.5000 - mae: 259.7493 - val_loss: 1.0080 - val_mae: 0.6064
Epoch 2/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 7582.3335 - mae: 45.4224 - val_loss: 1.0080 - val_mae: 0.6066
Epoch 3/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1923.8889 - mae: 15.7404 - val_loss: 1.0081 - val_mae: 0.6059
Epoch 4/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 913.8741 - mae: 8.5396 - val_loss: 1.0081 - val_mae: 0.6059
Epoch 5/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 467.9740 - mae: 5.2786 - val_loss: 1.0081 - val_mae: 0.6059
Epoch 6/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 272.1166 - mae: 3.5029 - val_loss: 1.0081 - val_mae: 0.6061
Epoch 7/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 160.1092 - mae: 2.4588 - val_loss: 1.0081 - val_mae: 0.6062
Epoch 8/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 130.4043 - mae: 1.9830 - val_loss: 1.0080 - val_mae: 0.6065
Epoch 9/100
381/381 ━

2025/11/09 16:57:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:57:24 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during keras autologging: BAD_REQUEST: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: metrics.key, metrics.timestamp, metrics.step, metrics.run_uuid, metrics.value, metrics.is_nan
[SQL: INSERT INTO metrics ("key", value, timestamp, step, is_nan, run_uuid) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('validation_mae', 0.606444776058197, 1762696442173, 0, 0, '60953bcd05d740a5a3b70405450fcd71'), ('validation_mae', 0.6065647602081299, 1762696455146, 0, 0, '60953bcd05d740a5a3b70405450fcd71'), ('validation_mae', 0.6059451103210449, 1762696456544, 0, 0, '60953bcd05d740a5a3b70405450fcd71'), ('validation_mae', 0.6059141755104065, 1762696458061, 0, 0, '60953bcd05d740a5a3b7

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
🏃 View run third_run at: http://localhost:5000/#/experiments/1/runs/60953bcd05d740a5a3b70405450fcd71
🧪 View experiment at: http://localhost:5000/#/experiments/1
